# Figure 2 — simulation results

A clean rewrite of `figure2_revision.ipynb`.

**How to use it.** Say which parameters you want; the notebook finds the files.

```r
m <- build_manifest()                       # what is on disk, and with what settings
d <- load_sims(sim_category = 'A',          # -> list(enloc, traits, leads, vars, manifest)
               gwas_mult = 5, gtex_mult = 20,
               causal_min_maf = 0.001)
plot_enloc_outcomes(d, gtex_cat = 'gtex', rcp_cutoff = 0.5)
```

Nothing is hardcoded to one arm: the species prefix (`h`/`c`), the phenotype
scalings, the causal-MAF floor and the panel inventory are all read off the files,
and the fine-mapping floor and LD threshold off the arm directory name.

**What changed from `figure2_revision.ipynb`** (which is left untouched):

- `get_dfs()` assumed the nested `<base>/<CAT>/<CAT><i>` layout; every current arm is
  flat (`<base>/<CAT><i>`), which is why its last cell errored.
- `read_traits()` globbed `gwas_scaling_35_pheno.sbams`, so it only ever worked on 35x
  arms. The scaling now comes from the manifest, and only column 2 of the sbams file is
  read (~210 MB of phenotype matrices no longer parsed).
- `paste0(sim_category, str(i))` — R's `str()` prints and returns `NULL`, so `sim` was
  `"A"` for *every* replicate and `group_by(gwas, sim)` merged them. Runs are now keyed
  on `run_id`.
- `read_lead_snps()` derived `category` from `gsub('_.*','',file)`, which returns a
  truncated absolute O2 path now that plink's column 1 is a full path.
- The `m4` branch is gone: `m4_marks.tsv` exists nowhere under `simulation_data/`, and
  the positive-selection categories F and G have no fetched data. It also contained a
  call to `cowplot(...)`, which is not a function.
- ~200 lines of commented-out copy-paste, two cells referencing an undefined `eenloc`,
  and ~25 ad-hoc exploration cells removed.

## Setup

In [ ]:
library(dplyr)
library(stringr)
library(tidyr)
library(ggplot2)
library(cowplot)
suppressPackageStartupMessages(library(data.table))

fig_size = function(width, heigth){
    options(repr.plot.width = width, repr.plot.height = heigth)
}

color_key = list(
    'human gwas' = 'lightpink1',
    'human eqtls' = 'firebrick1',
    'human coloc' = 'firebrick4',
    'pig gwas' = 'lightskyblue1',
    'pig eqtls' = 'royalblue1',
    'pig coloc' = 'navyblue',
    'cattle gwas' = 'mediumpurple1',
    'cattle eqtls' = 'purple',
    'cattle coloc' = 'purple4',
    'human' = 'firebrick1',
    'pig' = 'royalblue1',
    'cattle' = 'purple'
)

# ---------------------------------------------------------------------------
# Where the fetched simulation output lives, and the vocabulary used to name it
# ---------------------------------------------------------------------------
SIM_ROOT = '/Users/noah/colocalization_humans_cattle_pigs/simulation_data'

# Panels. The eQTL panel size is encoded as a WORD, never as a number in any
# filename: gtex = 1000 individuals, gtex_smaller = 500, gtex_smallest = 250.
PANELS         = c('gwas', 'gtex', 'gtex_smaller', 'gtex_smallest')
GTEX_PANELS    = c('gtex', 'gtex_smaller', 'gtex_smallest')
DEFAULT_PANELS = c('gwas', 'gtex', 'gtex_smaller')
PANEL_N        = c(gwas = NA, gtex = 1000, gtex_smaller = 500, gtex_smallest = 250)

# Category letter -> demography, matching simulations/helpers/summarize_coloc.py.
# Only used as a cross-check; `species` in the manifest is read off the actual
# file prefix (h/c), which is authoritative.
CATEGORY_SPECIES = c(A = 'human', B = 'human', C = 'human', D = 'human',
                     E = 'cattle', F = 'cattle', G = 'cattle')

# Binning, defined once instead of being repeated in three places.
SELCO_BREAKS = -1 * c(Inf, 2**seq(-10, -15), 0, -Inf)
MAF_BREAKS   = seq(0, 0.5, 0.05)

## Finding the data

`build_manifest()` walks `simulation_data/`, skips anything named `OLD_*`, and returns one
row per replicate directory describing what it holds and what settings produced it.

Two parameters can only come from the **directory name**, never a filename:

- **`fm_min_maf`** — the `maf_<X>` component of a filename is the *causal* floor. A sibling
  arm with `fm_min_maf = 0.001` is queued whose stage-2 filenames will be byte-identical to
  `cmaf001_fm01_*` (see `PROVENANCE.txt` in that arm).
- **`fm_r2`** (`ld_ctrl`) — never appears in any fetched filename.

Where a value can be read from both the directory name and the filenames, the two are
cross-checked and a disagreement is warned about rather than silently resolved.

In [ ]:
# ===========================================================================
# Filename patterns.
#
# Panel names are prefixes of one another (gtex / gtex_smaller / gtex_smallest),
# so every pattern is ANCHORED and includes the token that follows the panel
# name. `^[ch]gtex_vars_` does not match `hgtex_smaller_vars_...`.
#
# The species prefix is `h` or `c`; it is matched, never constructed, which is
# what lets one set of patterns serve both species.
# ===========================================================================
pat_vars   = function(panel) paste0('^[ch]', panel, '_vars_gwas_[0-9]+_gtex_[0-9]+_maf_[0-9.eE+-]+\\.tsv$')
pat_traits = function(panel) paste0('^[ch]', panel, '_traits_gwas_[0-9]+_gtex_[0-9]+_maf_[0-9.eE+-]+\\.tsv$')
pat_sbams  = function(panel) paste0('^[ch]', panel, '_scaling_[0-9]+_pheno\\.sbams$')
pat_leads  = function(panel) paste0('^[ch]', panel, '_glm_lead_snps\\.tsv$')
pat_enloc  = function(panel) paste0('\\.[ch]', panel, '\\.enloc\\.sig\\.out$')

# MAF tokens in directory names are written without the leading "0.":
# cmaf001 -> 0.001, fm01 -> 0.01. The queued fm001 arm parses with no code change.
maf_token = function(digits) as.numeric(paste0('0.', digits))

# ---------------------------------------------------------------------------
# parse_arm(): read run parameters out of an arm DIRECTORY name.
#
# Two of these can only ever come from the directory:
#   fm_min_maf -- the `maf_<X>` component of a filename is the CAUSAL floor, not
#                 the fine-mapping floor. A sibling arm with fm_min_maf = 0.001
#                 is queued (simulations/submit_2Mb_r3_cmaf_fm001.sh) whose
#                 stage-2 filenames will be byte-identical to cmaf001_fm01_*.
#                 See simulation_data/cmaf001_fm01_g5t20_2Mb/PROVENANCE.txt.
#   fm_r2      -- ld_ctrl never appears in a fetched filename at all.
#
# Token scanning rather than whole-name matching, so a new arm parses without
# editing this function.
# ---------------------------------------------------------------------------
parse_arm = function(arm) {
    grab = function(re) {
        m = str_match(arm, re)
        if (is.na(m[1, 1])) NULL else m[1, -1]
    }
    out = list(fm_r2 = NA_real_, fm_min_maf = NA_real_, causal_min_maf = NA_real_,
               gwas_mult = NA_real_, gtex_mult = NA_real_, region = NA_character_,
               fm_r2_source = 'unset')

    if (!is.null(g <- grab('(?:^|_)r2_([0-9]+)(?:_|$)'))) {
        out$fm_r2 = as.numeric(g[1]) / 100
        out$fm_r2_source = 'dirname'
    } else {
        # ld_ctrl 0.75 is the pipeline default and is implicit on O2 (only the
        # 0.25 arm gets an `_r2_0_25` suffix). Recorded as a default, not a read.
        out$fm_r2 = 0.75
        out$fm_r2_source = 'pipeline default'
    }
    if (!is.null(g <- grab('(?:^|_)cmaf([0-9]+)(?:_|$)'))) out$causal_min_maf = maf_token(g[1])
    if (!is.null(g <- grab('(?:^|_)fm([0-9]+)(?:_|$)')))   out$fm_min_maf     = maf_token(g[1])
    if (!is.null(g <- grab('(?:^|_)g([0-9]+)t([0-9]+)(?:_|$)'))) {
        out$gwas_mult = as.numeric(g[1]); out$gtex_mult = as.numeric(g[2])
    }
    if (!is.null(g <- grab('(?:^|_)([0-9]+)x(?:_|$)'))) {
        out$gwas_mult = as.numeric(g[1]); out$gtex_mult = as.numeric(g[1])
    }
    if (!is.null(g <- grab('(?:^|_)([0-9]+)(Mb|kb)(?:_|$)'))) out$region = paste0(g[1], g[2])
    out
}

# ---------------------------------------------------------------------------
# read_provenance(): pull the settings a PROVENANCE.txt records, for cross-check.
# Best effort -- returns an empty list if the file is missing or unparseable.
# ---------------------------------------------------------------------------
read_provenance = function(path) {
    if (is.na(path) || !file.exists(path)) return(list())
    txt = paste(readLines(path, warn = FALSE), collapse = '\n')
    num = function(re) {
        m = str_match(txt, re)
        if (is.na(m[1, 1])) NA_real_ else as.numeric(m[1, 2])
    }
    list(causal_min_maf = num('causal_min_maf\\s+([0-9.eE+-]+)'),
         fm_min_maf     = num('fm_min_maf\\s+([0-9.eE+-]+)'),
         min_maf        = num('\\bmin_maf\\s+([0-9.eE+-]+)'),
         fm_r2          = num('ld_ctrl\\s+([0-9.eE+-]+)'),
         gwas_mult      = num('gwas_scaling\\s+([0-9]+)'),
         gtex_mult      = num('gtex_scaling\\s+([0-9]+)'))
}

# ---------------------------------------------------------------------------
# scan_run_dir(): inventory ONE replicate directory.
# Everything that can be read from the files is read from the files.
# ---------------------------------------------------------------------------
scan_run_dir = function(dir, arm, leaf, arm_par) {
    files = list.files(dir)
    first = function(x) if (length(x)) x[1] else NA_character_

    have = function(patfun) {
        PANELS[vapply(PANELS, function(p) any(str_detect(files, patfun(p))), logical(1))]
    }
    p_vars   = have(pat_vars);   p_sbams = have(pat_sbams)
    p_leads  = have(pat_leads);  p_enloc = have(pat_enloc)
    p_traits = have(pat_traits)

    # Species from the prefix actually present, not from the category letter.
    sp_char = unique(str_match(files, '^([ch])(?:gwas|gtex)')[, 2])
    sp_char = sp_char[!is.na(sp_char)]
    species = if (length(sp_char) == 1) c(h = 'human', c = 'cattle')[[sp_char]] else NA_character_

    # gwas_mult / gtex_mult / causal_min_maf from any *_vars_* filename.
    vfile = first(files[str_detect(files, '_vars_gwas_[0-9]+_gtex_[0-9]+_maf_')])
    vm = if (is.na(vfile)) rep(NA_character_, 3) else
         str_match(vfile, '_vars_gwas_([0-9]+)_gtex_([0-9]+)_maf_([0-9.eE+-]+)\\.tsv$')[1, -1]

    # fastEnloc output basename ("human", "cattle_baseline_from_midpoint", ...).
    efile = first(files[str_detect(files, '\\.enloc\\.sig\\.out$')])
    ebase = if (is.na(efile)) NA_character_ else str_match(efile, '^(.+?)\\.[ch]gtex')[1, 2]

    reason = if (length(files) == 0) 'empty'
        else if (length(p_vars) == 0) 'no *_vars_* files'
        else if (length(p_enloc) == 0) 'no *.enloc.sig.out'
        else if (length(p_leads) == 0) 'no *_glm_lead_snps.tsv'
        else NA_character_

    data.frame(
        arm            = arm,
        sim            = leaf,
        sim_category   = str_match(leaf, '^([A-G])')[1, 2],
        replicate      = as.integer(str_match(leaf, '^[A-G]([0-9]+)$')[1, 2]),
        run_id         = paste0(arm, '/', leaf),
        species        = species,
        basename       = ebase,
        gwas_mult      = as.numeric(vm[1]),
        gtex_mult      = as.numeric(vm[2]),
        causal_min_maf = as.numeric(vm[3]),
        fm_min_maf     = arm_par$fm_min_maf,
        fm_r2          = arm_par$fm_r2,
        region         = arm_par$region,
        panels_vars    = paste(p_vars,   collapse = ','),
        panels_sbams   = paste(p_sbams,  collapse = ','),
        panels_leads   = paste(p_leads,  collapse = ','),
        panels_enloc   = paste(p_enloc,  collapse = ','),
        panels_traits  = paste(p_traits, collapse = ','),
        n_files        = length(files),
        ok             = is.na(reason),
        reason         = reason,
        dir            = dir,
        stringsAsFactors = FALSE
    )
}

# ---------------------------------------------------------------------------
# build_manifest(): one row per replicate directory under `root`.
# Handles both the current flat layout (<arm>/<CAT><i>) and the legacy nested
# one (<arm>/<CAT>/<CAT><i>). Skips anything named OLD_*.
# ---------------------------------------------------------------------------
build_manifest = function(root = SIM_ROOT, quiet = FALSE) {
    arms = list.dirs(root, recursive = FALSE, full.names = FALSE)
    arms = arms[!str_starts(arms, 'OLD_')]

    rows = list(); warn = character(0)
    for (arm in arms) {
        arm_dir  = file.path(root, arm)
        arm_par  = parse_arm(arm)
        prov_path = file.path(arm_dir, 'PROVENANCE.txt')
        prov = read_provenance(if (file.exists(prov_path)) prov_path else NA_character_)
        if (!is.null(prov$fm_min_maf) && !is.na(prov$fm_min_maf)) {
            # PROVENANCE.txt is authoritative for the floors the dirname cannot carry.
            if (is.na(arm_par$fm_min_maf)) arm_par$fm_min_maf = prov$fm_min_maf
        }
        if (!is.null(prov$fm_r2) && !is.na(prov$fm_r2) && arm_par$fm_r2_source != 'dirname') {
            arm_par$fm_r2 = prov$fm_r2
            arm_par$fm_r2_source = 'PROVENANCE.txt'
        }

        # Candidate leaves: direct children that look like <CAT><i>, plus one
        # more level down for the legacy <CAT>/<CAT><i> nesting.
        for (lvl1 in list.dirs(arm_dir, recursive = FALSE, full.names = FALSE)) {
            if (str_detect(lvl1, '^[A-G][0-9]+$')) {
                rows[[length(rows) + 1]] = scan_run_dir(file.path(arm_dir, lvl1), arm, lvl1, arm_par)
            } else if (str_detect(lvl1, '^[A-G]$')) {
                kids = list.dirs(file.path(arm_dir, lvl1), recursive = FALSE, full.names = FALSE)
                kids = kids[str_detect(kids, '^[A-G][0-9]+$')]
                if (length(kids) == 0) {
                    rows[[length(rows) + 1]] = scan_run_dir(file.path(arm_dir, lvl1), arm, paste0(lvl1, '0'), arm_par) %>%
                        mutate(sim = lvl1, replicate = NA_integer_, run_id = paste0(arm, '/', lvl1),
                               ok = FALSE, reason = 'empty')
                }
                for (k in kids) {
                    rows[[length(rows) + 1]] = scan_run_dir(file.path(arm_dir, lvl1, k), arm, k, arm_par)
                }
            }
        }

        # Cross-check dirname / PROVENANCE against what the filenames say.
        if (length(rows)) {
            got = bind_rows(rows) %>% filter(arm == !!arm, ok)
            chk = function(field, want, label) {
                if (is.null(want) || is.na(want) || nrow(got) == 0) return(invisible())
                bad = got[[field]][!is.na(got[[field]]) & abs(got[[field]] - want) > 1e-12]
                if (length(bad)) warn <<- c(warn, sprintf(
                    '%s: %s from filenames is %s but %s says %s',
                    arm, field, paste(unique(bad), collapse = '/'), label, want))
            }
            chk('gwas_mult',      arm_par$gwas_mult,   'the directory name')
            chk('gtex_mult',      arm_par$gtex_mult,   'the directory name')
            chk('causal_min_maf', arm_par$causal_min_maf, 'the directory name')
            chk('gwas_mult',      prov$gwas_mult,      'PROVENANCE.txt')
            chk('gtex_mult',      prov$gtex_mult,      'PROVENANCE.txt')
            chk('causal_min_maf', prov$causal_min_maf, 'PROVENANCE.txt')
        }
    }

    m = bind_rows(rows) %>%
        mutate(species_expected = unname(CATEGORY_SPECIES[sim_category])) %>%
        arrange(arm, sim_category, replicate)

    mism = m %>% filter(ok, !is.na(species), species != species_expected)
    if (nrow(mism)) warn = c(warn, sprintf('%s: files are %s but category %s implies %s',
                                           mism$run_id, mism$species, mism$sim_category,
                                           mism$species_expected))
    m$species_expected = NULL

    for (w in warn) warning(w, call. = FALSE)
    if (!quiet) print_manifest(m)
    invisible(m)
}

# Short headers so the summary fits an 80-column notebook cell (new = old).
MANIFEST_SHOW = c(arm = 'arm', species = 'species', gwas_x = 'gwas_mult',
                  gtex_x = 'gtex_mult', cmaf = 'causal_min_maf',
                  fm = 'fm_min_maf', r2 = 'fm_r2')

print_manifest = function(m) {
    good = m %>% filter(ok); bad = m %>% filter(!ok)
    cat(sprintf('%d loadable run(s) in %d arm(s):\n\n',
                nrow(good), length(unique(good$arm))))
    if (nrow(good)) {
        good %>%
            group_by(across(all_of(unname(MANIFEST_SHOW)))) %>%
            summarize(runs = paste(sim, collapse = ','), .groups = 'drop') %>%
            rename(all_of(MANIFEST_SHOW)) %>%
            relocate(runs, .after = arm) %>%
            as.data.frame() %>% print(row.names = FALSE)
    }
    if (nrow(bad)) {
        cat(sprintf('\n%d directory/ies present but not loadable:\n\n', nrow(bad)))
        bad %>% select(arm, sim, n_files, reason) %>% as.data.frame() %>%
            print(row.names = FALSE)
    }
    cat('\nColumns available for filtering:\n  ',
        paste(setdiff(names(m), c('dir', 'reason')), collapse = ', '), '\n', sep = '')
    invisible(m)
}

# Memoised accessor, so repeated load_sims() calls do not re-walk the tree.
.manifest_cache = new.env(parent = emptyenv())
get_manifest = function(root = SIM_ROOT, refresh = FALSE) {
    if (refresh || is.null(.manifest_cache[[root]])) {
        .manifest_cache[[root]] = build_manifest(root, quiet = TRUE)
    }
    .manifest_cache[[root]]
}

## Selecting runs

In [ ]:
# ---------------------------------------------------------------------------
# select_runs(): pick manifest rows by parameter. Every filter is optional;
# NULL means "do not filter on this". A run whose value is NA for a field never
# matches a concrete value for that field -- e.g. the r2_* arms record no
# fm_min_maf (it is not in their directory name), so fm_min_maf = 0.01 excludes
# them rather than silently assuming.
# ---------------------------------------------------------------------------
.match_any = function(x, val) {
    if (is.numeric(x)) {
        vapply(x, function(v) !is.na(v) && any(abs(v - as.numeric(val)) < 1e-12),
               logical(1))
    } else {
        !is.na(x) & x %in% val
    }
}

select_runs = function(manifest = NULL, arm = NULL, sim_category = NULL,
                       species = NULL, replicate = NULL, sim = NULL,
                       gwas_mult = NULL, gtex_mult = NULL, causal_min_maf = NULL,
                       fm_min_maf = NULL, fm_r2 = NULL, region = NULL,
                       run_id = NULL, include_unloadable = FALSE, quiet = FALSE) {
    if (is.null(manifest)) manifest = get_manifest()
    filters = list(arm = arm, sim_category = sim_category, species = species,
                   replicate = replicate, sim = sim, gwas_mult = gwas_mult,
                   gtex_mult = gtex_mult, causal_min_maf = causal_min_maf,
                   fm_min_maf = fm_min_maf, fm_r2 = fm_r2, region = region,
                   run_id = run_id)
    filters = filters[!vapply(filters, is.null, logical(1))]

    keep = rep(TRUE, nrow(manifest))
    for (nm in names(filters)) keep = keep & .match_any(manifest[[nm]], filters[[nm]])
    sel = manifest[keep, , drop = FALSE]

    if (nrow(sel) == 0) {
        shown = if (length(filters)) paste(names(filters), vapply(filters, function(v)
            paste(v, collapse = '/'), character(1)), sep = ' = ', collapse = ', ')
            else '(no filters)'
        cat('Nothing matched. Available loadable runs:\n\n')
        print_manifest(manifest)
        stop('no runs matched: ', shown, call. = FALSE)
    }
    if (!include_unloadable) {
        skipped = sel %>% filter(!ok)
        if (nrow(skipped)) {
            warning(sprintf('skipping %d unloadable run(s): %s',
                            nrow(skipped),
                            paste(sprintf('%s (%s)', skipped$run_id, skipped$reason),
                                  collapse = ', ')), call. = FALSE)
        }
        sel = sel %>% filter(ok)
        if (nrow(sel) == 0) stop('every matching run is unloadable', call. = FALSE)
    }
    if (!quiet) cat(sprintf('matched %d run(s): %s\n', nrow(sel),
                            paste(sel$run_id, collapse = ', ')))
    sel
}

# Does this run have file family `family` for `panel`?
has_panel = function(run, family, panel) {
    panel %in% strsplit(run[[paste0('panels_', family)]], ',')[[1]]
}

## Reading the files

Four file families per replicate directory (`<sp>` is `h` or `c`):

| family | pattern | contents |
|---|---|---|
| vars | `<sp><panel>_vars_gwas_<G>_gtex_<T>_maf_<CAUSAL>.tsv` | every variant: `selco`, `daf`, `maf`, `Vs`, `beta` |
| sbams | `<sp><panel>_scaling_<S>_pheno.sbams` | DAP-G phenotypes; column 2 is the trait id |
| leads | `<sp><panel>_glm_lead_snps.tsv` | plink2 `--glm` lead SNP per trait, no header |
| enloc | `<basename>.<sp><panel>.enloc.sig.out` | fastEnloc signals with `RCP` / `LCP` |

Panel names are prefixes of one another (`gtex` / `gtex_smaller` / `gtex_smallest`), so every
pattern is anchored and includes the token that follows the panel name.

In [ ]:
# ---------------------------------------------------------------------------
# find_file(): resolve exactly one file, or stop with a useful message.
# (The old get_regex_file() printed and returned NULL, so a miss surfaced
# several frames later as an opaque read.table error.)
# ---------------------------------------------------------------------------
find_file = function(dir, pattern) {
    hits = list.files(dir, pattern = pattern)
    if (length(hits) == 0) {
        stop(sprintf('no file matching /%s/ in %s\n  directory contains: %s',
                     pattern, dir, paste(list.files(dir), collapse = ', ')),
             call. = FALSE)
    }
    if (length(hits) > 1) {
        stop(sprintf('%d files match /%s/ in %s: %s', length(hits), pattern, dir,
                     paste(hits, collapse = ', ')), call. = FALSE)
    }
    file.path(dir, hits[1])
}

# ---------------------------------------------------------------------------
# read_vars(): the true causal-variant table for one panel.
# Columns: id type selco daf time position maf Vs beta.  beta != 0 marks a
# causative variant; selco is already un-scaled by Q_scaling upstream.
# ---------------------------------------------------------------------------
read_vars = function(run, panel) {
    read.table(find_file(run$dir, pat_vars(panel)), header = TRUE, as.is = TRUE) %>%
        transmute(trait = paste0('tr', position),
                  position, time, daf, maf, selco, Vs, beta)
}

# Side-prefix a vars table for joining onto enloc (gwas_maf, gtex_selco, ...).
# Explicit names avoid the position.x / position.y that the old double
# left_join produced.
prefix_vars = function(v, side) {
    names(v) = ifelse(names(v) == 'trait', side, paste0(side, '_', names(v)))
    v
}

# ---------------------------------------------------------------------------
# read_lead_snps(): plink2 --glm output, concatenated with a leading filename
# column. No header. Column 1 is a full O2 path, so the trait id is taken from
# the embedded `*_glm.<trait>.glm.linear`, and `panel` comes from the argument
# rather than from parsing that path.
# ---------------------------------------------------------------------------
GLM_COLS = c('file', 'chr', 'pos', 'id', 'ref', 'alt', 'provisional_ref', 'a1',
             'omitted', 'a1_freq', 'test', 'obs_ct', 'beta', 'se', 't_stat',
             'p', 'errcode')

read_lead_snps = function(run, panel) {
    read.table(find_file(run$dir, pat_leads(panel)), sep = '\t',
               col.names = GLM_COLS, as.is = TRUE) %>%
        transmute(panel = panel,
                  trait = str_extract(file, 'tr[0-9]+'),
                  trait_pos = as.numeric(str_remove(str_extract(file, 'tr[0-9]+'), 'tr')),
                  chr, pos, id, ref, alt, a1, a1_freq, beta, se, p)
}

# ---------------------------------------------------------------------------
# read_enloc(): the joined colocalization table for one GTEx panel.
#
# Signal looks like `tr378663:2(@)tr647928:2`. The left side is the eQTL
# (GTEx) signal, the right the GWAS locus. fastEnloc merges GWAS loci that
# share SNPs into an underscore-joined list -- `tr634882:1_tr717151:1` -- and
# every constituent is credited (one row each), matching the previous
# behaviour. A signal whose SNPs fall in no GWAS locus has an empty right side
# and is dropped; the count is reported.
#
# NOTE: .enloc.sig.out is NOT tab-delimited. The header is tab-separated but
# data rows are space-padded with a tab only before LCP, so read.table must use
# its default whitespace separator.
# ---------------------------------------------------------------------------
read_enloc = function(run, panel) {
    raw = read.table(find_file(run$dir, pat_enloc(panel)), header = TRUE, as.is = TRUE)

    split = raw %>%
        separate_wider_delim(Signal, delim = '(@)', names = c('gtex', 'gwas'),
                             too_few = 'align_start', cols_remove = FALSE)
    n_no_locus = sum(is.na(split$gwas) | split$gwas == '')

    sig = split %>%
        filter(!is.na(gwas), gwas != '') %>%
        separate_longer_delim(gwas, delim = '_') %>%
        mutate(gtex = str_replace(gtex, ':[0-9]+', ''),
               gwas = str_replace(gwas, ':[0-9]+', ''))

    leads_gwas = read_lead_snps(run, 'gwas') %>%
        transmute(gwas = trait, min_snp_pos_gwas = pos, gwas_p = p)
    leads_gtex = read_lead_snps(run, panel) %>%
        transmute(gtex = trait, min_snp_pos_gtex = pos, gtex_p = p)

    # The full_joins are load-bearing: they add rows for GWAS traits that have
    # no enloc signal at all. Those get correct = NA and are classified
    # downstream as "Underpowered fine-mapping".
    e = sig %>%
        mutate(correct = !is.na(gwas) & gtex == gwas) %>%
        full_join(leads_gwas, by = 'gwas') %>%
        full_join(leads_gtex, by = 'gtex') %>%
        distinct() %>%
        left_join(prefix_vars(read_vars(run, panel), 'gtex'), by = 'gtex') %>%
        left_join(prefix_vars(read_vars(run, 'gwas'), 'gwas'), by = 'gwas') %>%
        mutate(
            selco_bin = cut(gwas_selco, breaks = SELCO_BREAKS, right = FALSE),
            maf_bin   = cut(gwas_maf,   breaks = MAF_BREAKS),
            dist_true_gwas_gtex = abs(as.numeric(str_remove(gwas, 'tr')) -
                                      as.numeric(str_remove(gtex, 'tr'))),
            dist_lead_gwas_gtex = abs(min_snp_pos_gwas - min_snp_pos_gtex)
        )
    attr(e, 'n_signals')  = nrow(raw)
    attr(e, 'n_no_locus') = n_no_locus
    e
}

# ---------------------------------------------------------------------------
# read_traits(): traits present in both the GWAS and the given panel.
#
# The sbams files are the only trait listing available in every arm (the
# *_traits_*.tsv family exists in the r2_* arms but not in cmaf001_*), and they
# total ~210 MB. Column 2 holds the trait id, so only that column is read.
# The scaling in the filename comes from the manifest, not a hardcoded 35.
# ---------------------------------------------------------------------------
sbams_traits = function(path) {
    unique(as.character(fread(path, sep = '\t', select = 2, header = FALSE,
                              showProgress = FALSE)[[1]]))
}

read_traits = function(run, panel) {
    g = sbams_traits(find_file(run$dir, pat_sbams('gwas')))
    p = sbams_traits(find_file(run$dir, pat_sbams(panel)))
    data.frame(trait = intersect(g, p), stringsAsFactors = FALSE)
}

# read_m4() and the `m4` argument are deliberately absent: m4_marks.tsv exists
# nowhere under simulation_data/, and the positive-selection categories F and G
# have no fetched data at all. Restore both together if that changes.

## Loading

In [ ]:
# Run-identifying columns stamped onto every returned dataframe, so plots can
# facet or group on any of them without a separate join.
RUN_COLS = c('run_id', 'arm', 'sim', 'sim_category', 'replicate', 'species',
             'gwas_mult', 'gtex_mult', 'causal_min_maf', 'fm_min_maf', 'fm_r2')

# ---------------------------------------------------------------------------
# load_sims(): the one function you call.
#
#   load_sims(sim_category = 'A', gwas_mult = 5, gtex_mult = 20,
#             causal_min_maf = 0.001)
#
# Filter arguments are those of select_runs(). Returns
#   list(enloc =, traits =, leads =, vars =, manifest =)
# with the same shape as the old get_dfs(), plus run-identifying columns.
#
# `enloc` keeps the column name `gtex_category`; `traits`, `leads` and `vars`
# keep `type` -- the names the existing plotting code filters on.
# ---------------------------------------------------------------------------
load_sims = function(..., panels = DEFAULT_PANELS, manifest = NULL, quiet = FALSE) {
    runs = select_runs(manifest = manifest, ..., quiet = quiet)

    bad_panel = setdiff(panels, PANELS)
    if (length(bad_panel)) {
        stop('unknown panel(s): ', paste(bad_panel, collapse = ', '),
             '. Known panels: ', paste(PANELS, collapse = ', '), call. = FALSE)
    }
    if ('gtex_smallest' %in% panels) {
        warning('gtex_smallest is a stage-2-only panel: it has no *_glm_lead_snps.tsv ',
                'and no *.enloc.sig.out, so `enloc` and `leads` will contain no rows ',
                'for it.', call. = FALSE)
    }

    L_enloc = list(); L_traits = list(); L_leads = list(); L_vars = list()
    n_sig = 0; n_no_locus = 0

    for (i in seq_len(nrow(runs))) {
        run  = runs[i, ]
        meta = run[, RUN_COLS, drop = FALSE]
        stamp = function(df) if (nrow(df) == 0) df else bind_cols(df, meta[rep(1, nrow(df)), ])

        for (panel in panels) {
            if (has_panel(run, 'vars', panel)) {
                L_vars[[length(L_vars) + 1]] = stamp(
                    read_vars(run, panel) %>%
                        mutate(type = panel,
                               gtex_n = unname(PANEL_N[panel]),
                               selco_bin = cut(selco, breaks = SELCO_BREAKS, right = FALSE),
                               maf_bin   = cut(maf,   breaks = MAF_BREAKS)))
            }
            if (has_panel(run, 'leads', panel)) {
                L_leads[[length(L_leads) + 1]] = stamp(
                    read_lead_snps(run, panel) %>%
                        mutate(type = panel, gtex_n = unname(PANEL_N[panel])) %>%
                        select(-panel))
            }
            if (has_panel(run, 'sbams', panel)) {
                L_traits[[length(L_traits) + 1]] = stamp(
                    read_traits(run, panel) %>%
                        mutate(type = panel, gtex_n = unname(PANEL_N[panel])))
            }
            # There is no GWAS-vs-GWAS fastEnloc run, so enloc is GTEx panels only.
            if (panel %in% GTEX_PANELS && has_panel(run, 'enloc', panel)) {
                e = read_enloc(run, panel)
                n_sig      = n_sig + attr(e, 'n_signals')
                n_no_locus = n_no_locus + attr(e, 'n_no_locus')
                L_enloc[[length(L_enloc) + 1]] = stamp(
                    e %>% mutate(gtex_category = panel, gtex_n = unname(PANEL_N[panel])))
            }
        }
    }

    if (!quiet && n_sig > 0) {
        cat(sprintf('[sig.out] %d signal(s) read; %d (%.1f%%) fell in no GWAS locus and were dropped\n',
                    n_sig, n_no_locus, 100 * n_no_locus / n_sig))
    }

    out = list(enloc  = bind_rows(L_enloc),
               traits = bind_rows(L_traits),
               leads  = bind_rows(L_leads),
               vars   = bind_rows(L_vars),
               manifest = runs)
    if (!quiet) {
        cat(sprintf('rows: enloc %d | traits %d | leads %d | vars %d\n',
                    nrow(out$enloc), nrow(out$traits), nrow(out$leads), nrow(out$vars)))
    }
    out
}

## Plot helpers

In [ ]:
# ---------------------------------------------------------------------------
# plot_enloc_outcomes(): classify every GWAS trait into one of four outcomes
# and show the breakdown by GWAS MAF bin.
#
#   Underpowered fine-mapping   correct is NA -- the trait has no enloc signal
#   Underpowered colocalization RCP at or below the cutoff
#   True positive               the eQTL and GWAS trait ids match
#   False positive              they do not
#
# One row is kept per GWAS trait per RUN, taking the best available evidence
# (NA-correct first, then correct, then highest RCP). Grouping is on run_id,
# not sim: a trait id is a causal-variant position and can recur across
# replicates and arms, so grouping on anything coarser silently merges runs.
# ---------------------------------------------------------------------------
plot_enloc_outcomes = function(dfs, gtex_cat = 'gtex', rcp_cutoff = 0.5,
                               p_cutoff = 1, prop = FALSE, rtrn = FALSE, prnt = TRUE) {
    tmp = dfs$enloc %>%
        filter(gtex_category == gtex_cat) %>%
        arrange(desc(is.na(correct)), desc(correct), desc(is.na(RCP)), desc(RCP)) %>%
        group_by(gwas, run_id) %>%
        slice(1) %>%
        ungroup() %>%
        mutate(RCP = ifelse(is.na(RCP), 0, RCP),
               outcome = ifelse(is.na(correct), 'Underpowered fine-mapping',
                         ifelse(RCP <= rcp_cutoff, 'Underpowered colocalization',
                         ifelse(correct, 'True positive', 'False positive')))) %>%
        filter(!is.na(maf_bin), gwas_p < p_cutoff)

    tot = nrow(tmp)
    cat(sprintf('%s, RCP > %g, GWAS p < %g: %d GWAS trait(s) over %d run(s)\n',
                gtex_cat, rcp_cutoff, p_cutoff, tot, n_distinct(tmp$run_id)))
    counts = tmp %>% summarize(tp  = sum(outcome == 'True positive'),
                               fp  = sum(outcome == 'False positive'),
                               ufm = sum(outcome == 'Underpowered fine-mapping'),
                               uc  = sum(outcome == 'Underpowered colocalization'))
    if (prop) counts = counts %>% mutate(across(everything(), ~ .x / tot))
    print(counts)

    p1 = tmp %>%
        ggplot(aes(x = maf_bin, fill = outcome)) +
        geom_bar(stat = 'count') + theme_classic() +
        xlab('GWAS causal-variant MAF') + ylab('GWAS traits') +
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1))

    if (prnt) { fig_size(7, 5); print(p1) }
    if (rtrn) return(list(plot = p1, data = tmp, counts = counts))
    invisible(NULL)
}

# ---------------------------------------------------------------------------
# plot_selco_af(): selection-coefficient / allele-frequency diagnostics for the
# GWAS panel's variant table.
# ---------------------------------------------------------------------------
plot_selco_af = function(dfs, min_maf = 0, rtrn = FALSE, prnt = TRUE) {
    tmp = dfs$vars %>%
        filter(type == 'gwas', maf >= min_maf) %>%
        select(run_id, trait, selco, maf, maf_bin, selco_bin, daf) %>%
        distinct()

    p1 = tmp %>% ggplot(aes(x = selco_bin, y = maf)) +
        geom_boxplot() + theme_classic() +
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1))
    p2 = tmp %>% ggplot(aes(x = maf)) + geom_histogram(bins = 30) + theme_classic() +
        ggtitle('Distribution of all allele frequencies')
    p3 = tmp %>% filter(selco == 0) %>% ggplot(aes(x = maf)) +
        geom_histogram(bins = 30) + theme_classic() +
        ggtitle('Distribution of neutral allele frequencies')
    p4 = tmp %>% filter(selco != 0) %>% ggplot(aes(x = maf)) +
        geom_histogram(bins = 30) + theme_classic() +
        ggtitle('Distribution of non-neutral allele frequencies')

    if (prnt) { fig_size(12, 10); print(plot_grid(p1, p2, p3, p4, labels = 'AUTO', align = 'vh', ncol = 2)) }
    if (rtrn) return(list(p1, p2, p3, p4))
    invisible(NULL)
}

# Figure 2A — demographic histories

In [ ]:
cattle_demo = data.frame(
    species = 'cattle',
    gen_ago = c(44000, 33154, 3354, 2354, 1754, 654, 454, 154, 24, 18, 12, 6, 3),
    Ne = c(6.2e4, 1.7e4, 1e4, 7e3, 3.5e3, 2.5e3, 2e3, 1.5e3, 1e3, 350, 250, 120, 90)
)
cattle_demo$years_ago = cattle_demo$gen_ago * 5

anc_pop = 7300
anc_gen_ago = 8800
ooa_pop = 2100
ooa_gen_ago = 5600
eu_as_gen_ago = 848
eu_as_pop = 1000
growth_rate = 0.004

human_ne = function(gen_ago) {
    if(gen_ago > ooa_gen_ago) {
        return(anc_pop)
    } else if (gen_ago <= ooa_gen_ago && gen_ago > eu_as_gen_ago) {
        return(ooa_pop)
    } else if (gen_ago <= eu_as_gen_ago) {
        return(eu_as_pop * (1+growth_rate)^(eu_as_gen_ago - gen_ago))
    }
}

human_demo = data.frame(
    species = 'human',
    gen_ago = c(seq(anc_gen_ago, 500, -100), seq(490, 0, -10))
)
human_demo$Ne = sapply(human_demo$gen_ago, human_ne)
human_demo$years_ago = human_demo$gen_ago * 25

demos = rbind(cattle_demo, human_demo)

# This is just a fast way to get the fill correct
shifted_cattle = cattle_demo[2:nrow(cattle_demo), c(1,4)]
shifted_cattle$years_ago = shifted_cattle$years_ago + 1
shifted_cattle$Ne = cattle_demo$Ne[1:(nrow(cattle_demo)-1)]

shifted_human = human_demo[2:nrow(human_demo), c(1,4)]
shifted_human$years_ago = shifted_human$years_ago + 1
shifted_human$Ne = human_demo$Ne[1:(nrow(human_demo)-1)]

boxes = rbind(
    demos %>% select(species, years_ago, Ne),
    shifted_cattle,
    shifted_human
)

fig_size(10,5)

# I have have this double plotted stuff to make part of the box paler
boxes = boxes %>% mutate(alp = ifelse(species=='cattle', 0.2, 0.5))
boxes2 = boxes %>% filter(species=='cattle') %>% mutate(Ne = ifelse(Ne == 62000, 17000, Ne), alp = 0.3)

boxes %>%
arrange(species, desc(years_ago)) %>%
ggplot(aes(x=years_ago, y=Ne, color=species, fill=species)) +
    scale_x_reverse() + geom_step(show.legend=F) + theme_classic() +
    xlab('Years ago') + ylab('Effective population size (Ne)') + theme(legend.title=element_blank()) +
    geom_area(position='identity', aes(alpha=alp)) +
    geom_area(data=boxes2, aes(x=years_ago, y=Ne, alpha=alp), color=NA, fill=color_key[['cattle']], show.legend=F) +
    scale_color_manual(values=c('cattle'=color_key[['cattle']], 'human'=color_key[['human']]), labels=c('Cattle demography', 'Human (CEU) demography')) +
    scale_fill_manual(values=c('cattle'=color_key[['cattle']], 'human'=color_key[['human']]), labels=c('Cattle demography', 'Human (CEU) demography')) +
    scale_alpha(guide='none') +
    guides(fill = guide_legend(override.aes = list(alpha = 0.5)))

# Worked examples

### What is on disk?

In [ ]:
m <- build_manifest()

### Set 2: GWAS 5x, GTEx 20x, causal floor 0.001, fine-mapping and GLM floor 0.01, r2 0.75

Five human replicates (A1–A5) and five cattle (E1–E5). This is the arm described by
`simulation_data/cmaf001_fm01_g5t20_2Mb/PROVENANCE.txt`.

In [ ]:
Arare <- load_sims(sim_category = 'A', gwas_mult = 5, gtex_mult = 20, causal_min_maf = 0.001)
Erare <- load_sims(sim_category = 'E', gwas_mult = 5, gtex_mult = 20, causal_min_maf = 0.001)

In [ ]:
plot_enloc_outcomes(Arare, gtex_cat = 'gtex', rcp_cutoff = 0.5)
plot_enloc_outcomes(Erare, gtex_cat = 'gtex', rcp_cutoff = 0.5)

The 1000- and 500-individual eQTL panels, at both RCP thresholds. `tp` here reproduces
`enloc_pow_rcp50` / `enloc_pow_rcp90` in `simulation_data/coloc_summary_round3.tsv` exactly
(A: 32/26 and 21/17; E: 79/63 and 64/49), which is a useful check that the joins are right.
`fp` is not expected to match: this notebook keeps one best row per GWAS trait, while
`summarize_coloc.py` scores trait pairs.

In [ ]:
for (gc in c('gtex', 'gtex_smaller')) {
    for (rcp in c(0.5, 0.9)) {
        plot_enloc_outcomes(Arare, gtex_cat = gc, rcp_cutoff = rcp, prnt = FALSE)
        plot_enloc_outcomes(Erare, gtex_cat = gc, rcp_cutoff = rcp, prnt = FALSE)
    }
}

### Allele-frequency and selection diagnostics

The cmaf arm is the only one with both species fetched locally, so it is the only place
this comparison can be made from raw files.

In [ ]:
plot_selco_af(Arare)
plot_selco_af(Erare)

### Set 1: the multiplier grid

Only cattle (`E1`) was fetched for these arms — every `A1` directory is present but empty,
so `load_sims()` warns and skips it.

In [ ]:
grid <- load_sims(arm = c('r2_75_1x_2Mb', 'r2_75_5x_2Mb', 'r2_75_10x_2Mb',
                          'r2_75_20x_2Mb', 'r2_75_35x_2Mb'))
grid$enloc %>%
    count(arm, gwas_mult, gtex_category) %>%
    tidyr::pivot_wider(names_from = gtex_category, values_from = n)